In [1]:
from pathlib import Path
import pandas as pd
import re


import warnings
warnings.filterwarnings('ignore')

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold, GridSearchCV
from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix,
                             roc_auc_score, precision_recall_curve, ConfusionMatrixDisplay)
import joblib


import joblib, os
import numpy as np
import pandas as pd
import torch

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from torch.optim.lr_scheduler import ReduceLROnPlateau

import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier
import shap
import phate
from scipy.stats import spearmanr

# Set reproducibility seeds
np.random.seed(42)
torch.manual_seed(42)
import random
random.seed(42)

In [2]:
# Load the raw tables
DATA_DIR = Path("/def-singhm/masoudk")
RNASEQ_PATH = 'rnaseq_data.tsv'
META_PATH = 'meta_data.tsv'

# Load metadata normally
meta_df = pd.read_csv(META_PATH, sep="\t")

# Read first row for gene names
header_df = pd.read_csv(RNASEQ_PATH, sep="\t", header=None, nrows=1, dtype=str)
gene_names = header_df.iloc[0, :].tolist()

# Read data rows (skip header)
rnaseq_df = pd.read_csv(RNASEQ_PATH, sep="\t", header=None, skiprows=1, dtype=str, engine='python', on_bad_lines='warn')

# Pad/truncate to match header length
if rnaseq_df.shape[1] > len(gene_names):
    rnaseq_df = rnaseq_df.iloc[:, :len(gene_names)]
elif rnaseq_df.shape[1] < len(gene_names):
    for i in range(len(gene_names) - rnaseq_df.shape[1]):
        rnaseq_df[len(rnaseq_df.columns)] = None

rnaseq_df.columns = gene_names

# Rename first column to "sample_id"
rnaseq_df = rnaseq_df.rename(columns={gene_names[0]: 'sample_id'})

print(f"RNA-seq shape: {rnaseq_df.shape}")

RNA-seq shape: (6606, 19311)


In [3]:
sorted(meta_df["histological_grade"].dropna().unique())

['G1',
 'G2',
 'G3',
 'G4',
 'GB',
 'GX',
 'High Grade',
 'Low Grade',
 '[Discrepancy]',
 '[Unknown]']

In [4]:
TCGA_ID_PATTERN = re.compile(r"TCGA-[A-Z0-9]+-[A-Z0-9]+-\d+")


def _clean_meta(df: pd.DataFrame) -> pd.DataFrame:
    meta = df.copy()
    meta.columns = (
        meta.columns.astype(str)
        .str.strip()
        .str.strip('"')
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )
    
    if 'sample' in meta.columns:
        meta['patient_id'] = meta['sample'].astype(str).str.strip().str.replace('"', '')

    
    meta = meta.dropna(subset=['patient_id']).drop_duplicates('patient_id')

    if "histological_grade" in meta.columns:
        meta["histological_grade"] = (
            meta["histological_grade"].astype(str).str.strip().str.upper().replace({"": pd.NA})
        )

    return meta


def _clean_rnaseq(df: pd.DataFrame) -> pd.DataFrame:
    """
    RNA-seq data structure:
    - Header row: gene names (19311 fields)
    - Data rows: TCGA sample ID in col 1, then 19310 expression values
    """
    # Extract gene names from header
    gene_names = df.columns.astype(str).str.strip().str.replace('"', '')
    
    # Extract sample IDs from first column (no normalization)
    sample_ids = df.iloc[:, 0].astype(str).str.strip().str.replace('"', '')
    
    # Extract expression data: columns 2 onwards
    expr_data = df.iloc[:, 1:].copy()
    expr_data.columns = gene_names[:len(expr_data.columns)]
    expr_data = expr_data.apply(pd.to_numeric, errors="coerce")
    
    # Create dataframe with sample_id and gene expressions
    rna_clean = pd.concat([
        sample_ids.reset_index(drop=True),
        expr_data.reset_index(drop=True)
    ], axis=1)
    
    rna_clean.columns = ['patient_id'] + list(expr_data.columns)
    
    # Remove any empty sample IDs
    rna_clean = rna_clean[rna_clean['patient_id'].str.len() > 0].dropna(subset=['patient_id'])
    
    rna_clean = rna_clean.set_index('patient_id')
    
    # Handle duplicate patients by averaging
    rna_final = rna_clean.groupby(level=0).mean()
    rna_final = rna_final.T  # genes x patients
    rna_final['gene_id'] = rna_final.index
    rna_final = rna_final[['gene_id'] + [c for c in rna_final.columns if c != 'gene_id']]
    rna_final = rna_final.reset_index(drop=True)
    
    return rna_final


clean_meta_df = _clean_meta(meta_df)
clean_rnaseq_df = _clean_rnaseq(rnaseq_df)

rnaseq_patient_ids = set(clean_rnaseq_df.columns) - {"gene_id"}
matched_patient_ids = sorted(set(clean_meta_df["patient_id"]) & rnaseq_patient_ids)

clean_meta_df = clean_meta_df[clean_meta_df["patient_id"].isin(matched_patient_ids)].reset_index(drop=True)
clean_rnaseq_df = clean_rnaseq_df[["gene_id"] + matched_patient_ids]

print(f"Matched {len(matched_patient_ids)} patients across both tables")
print(f"Metadata: {len(clean_meta_df)} rows")
print(f"RNA-seq: {len(clean_rnaseq_df)} genes × {len(matched_patient_ids)} samples")
clean_meta_df.head()

Matched 6606 patients across both tables
Metadata: 6606 rows
RNA-seq: 19310 genes × 6606 samples


,meta_data.tsv,sample_type,project_id,rnaseqid,mutid,sample,x_patient,cancer.type.abbreviation,age_at_initial_pathologic_diagnosis,gender,...,os.time,dss,dss.time,dfi,dfi.time,pfi,pfi.time,redaction,x_primary_disease,patient_id
0,1.0,Primary Tumor,TCGA-GBM,TCGA-02-0047-01A,TCGA-02-0047-01,TCGA-02-0047-01,TCGA-02-0047,GBM,78.0,MALE,...,448.0,1.0,448.0,NaN,NaN,1.0,57.0,NaN,glioblastoma multiforme,TCGA-02-0047-01
1,1.0,Primary Tumor,TCGA-GBM,TCGA-02-0055-01A,TCGA-02-0055-01,TCGA-02-0055-01,TCGA-02-0055,GBM,62.0,FEMALE,...,76.0,1.0,76.0,NaN,NaN,1.0,6.0,NaN,glioblastoma multiforme,TCGA-02-0055-01
2,1.0,Primary Tumor,TCGA-GBM,TCGA-02-2483-01A,TCGA-02-2483-01,TCGA-02-2483-01,TCGA-02-2483,GBM,43.0,MALE,...,466.0,0.0,466.0,NaN,NaN,0.0,466.0,NaN,glioblastoma multiforme,TCGA-02-2483-01
3,1.0,Primary Tumor,TCGA-GBM,TCGA-02-2485-01A,TCGA-02-2485-01,TCGA-02-2485-01,TCGA-02-2485,GBM,53.0,MALE,...,470.0,0.0,470.0,NaN,NaN,1.0,186.0,NaN,glioblastoma multiforme,TCGA-02-2485-01
4,1.0,Primary Tumor,TCGA-GBM,TCGA-02-2486-01A,TCGA-02-2486-01,TCGA-02-2486-01,TCGA-02-2486,GBM,64.0,MALE,...,618.0,1.0,618.0,NaN,NaN,1.0,618.0,NaN,glioblastoma multiforme,TCGA-02-2486-01


In [5]:
meta_test = meta_df.copy()
cols_lower = (
    meta_test.columns.astype(str)
    .str.strip()
    .str.strip('"')
    .str.lower()
    .str.replace(" ", "_", regex=False)
)
meta_test.columns = cols_lower

id_candidates = ['sample', 'x_patient', 'rnaseqid', 'sample_type_id', 'mutid']

def _normalize_patient_id(val):
    if pd.isna(val):
        return pd.NA
    s = str(val).strip().replace('"', '')
    return s if s else pd.NA

def _select_patient_id_test(row):
    for col in id_candidates:
        val = row.get(col)
        normalized = _normalize_patient_id(val)
        print(f"    {col}: {val} -> {normalized}")
        if not pd.isna(normalized):
            return normalized
    return pd.NA

In [6]:
print("Meta patient_id):")
print(clean_meta_df['patient_id'].head().tolist())

print("\nRNA-seq patient_id:")
print(list(rnaseq_patient_ids)[:5])

print(f"\nMeta unique count: {len(set(clean_meta_df['patient_id']))}")
print(f"RNA-seq unique count: {len(rnaseq_patient_ids)}")

print("\nIntersection:")
intersection = set(clean_meta_df["patient_id"]) & rnaseq_patient_ids
print(f"Matched: {len(intersection)}")

Meta patient_id):
['TCGA-02-0047-01', 'TCGA-02-0055-01', 'TCGA-02-2483-01', 'TCGA-02-2485-01', 'TCGA-02-2486-01']

RNA-seq patient_id:
['TCGA-B6-A40C-01', 'TCGA-3A-A9IB-01', 'TCGA-13-0714-01', 'TCGA-AG-3587-01', 'TCGA-56-8305-01']

Meta unique count: 6606
RNA-seq unique count: 6606

Intersection:
Matched: 6606


In [11]:
print("=== DATASET BEFORE LABELING ===")
print(f"Metadata shape: {clean_meta_df.shape}")
print(f"RNA-seq shape: {clean_rnaseq_df.shape}")

# Map cancer grades to binary risk labels
# low risk: G1 / LOW GRADE
# high risk: G3 / G4 / HIGH GRADE
# (G2 and unknowns are excluded)
low_risk_grades = {'G1', 'G2', 'LOW GRADE'}
high_risk_grades = {'G3', 'G4', 'HIGH GRADE'}

def grade_to_risk_label(grade):
    """Map histological grade to binary risk label."""
    if pd.isna(grade):
        return pd.NA
    grade_str = str(grade).strip().upper()
    if grade_str in low_risk_grades:
        return 0  # Low risk
    elif grade_str in high_risk_grades:
        return 1  # High risk
    else:
        return pd.NA  # Unknown

clean_meta_df['risk_label'] = clean_meta_df['histological_grade'].apply(grade_to_risk_label)

# Filter to only samples with valid risk labels
valid_idx = clean_meta_df['risk_label'].notna()
labeled_meta_df = clean_meta_df[valid_idx].reset_index(drop=True)

# Filter RNA-seq to match
valid_patient_ids = set(labeled_meta_df['patient_id'])
labeled_rnaseq_df = clean_rnaseq_df[['gene_id'] + sorted([p for p in clean_rnaseq_df.columns if p in valid_patient_ids])].reset_index(drop=True)

print("\n=== DATASET AFTER LABELING ===")
print(f"Metadata shape: {labeled_meta_df.shape}")
print(f"RNA-seq shape: {labeled_rnaseq_df.shape}")
print(f"\nRisk label distribution:")
print(labeled_meta_df['risk_label'].value_counts().sort_index())
print(f"  0 (Low risk):  {(labeled_meta_df['risk_label'] == 0).sum()}")
print(f"  1 (High risk): {(labeled_meta_df['risk_label'] == 1).sum()}")

=== DATASET BEFORE LABELING ===
Metadata shape: (6606, 42)
RNA-seq shape: (19310, 6607)

=== DATASET AFTER LABELING ===
Metadata shape: (2662, 42)
RNA-seq shape: (19310, 2663)

Risk label distribution:
risk_label
0    1117
1    1545
Name: count, dtype: int64
  0 (Low risk):  1117
  1 (High risk): 1545


In [12]:
def stage_0_prepare_data(X, y, test_size=0.15, val_size=0.15, random_state=42, output_dir='deployment'):

    os.makedirs(output_dir, exist_ok=True)
    
    # Stratified split: train+val vs test
    splitter1 = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=random_state)
    train_val_idx, test_idx = next(splitter1.split(X, y))
    
    X_train_val = X[train_val_idx]
    y_train_val = y[train_val_idx]
    X_test = X[test_idx]
    y_test = y[test_idx]
    
    # Stratified split: train vs val
    val_size_adjusted = val_size / (1.0 - test_size)
    splitter2 = StratifiedShuffleSplit(n_splits=1, test_size=val_size_adjusted, random_state=random_state)
    train_idx, val_idx = next(splitter2.split(X_train_val, y_train_val))
    
    X_train = X_train_val[train_idx]
    y_train = y_train_val[train_idx]
    X_val = X_train_val[val_idx]
    y_val = y_train_val[val_idx]
    
    # Fit scaler on train, apply to all
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_val_scaled = scaler.transform(X_val)
    X_test_scaled = scaler.transform(X_test)
    
    # Save scaler
    scaler_path = os.path.join(output_dir, 'scaler.pkl')
    joblib.dump(scaler, scaler_path)
    
    result = {
        'X_train': X_train_scaled,
        'y_train': y_train,
        'X_val': X_val_scaled,
        'y_val': y_val,
        'X_test': X_test_scaled,
        'y_test': y_test,
        'scaler': scaler,
        'scaler_path': scaler_path,
        'n_features': X.shape[1],
    }
    
    print(f"STAGE 0: Data Prepared")
    print(f"  Train: {result['X_train'].shape} | Val: {result['X_val'].shape} | Test: {result['X_test'].shape}")
    print(f"  Label distribution:")
    print(f"    Train - Low: {(y_train==0).sum()}, High: {(y_train==1).sum()}")
    print(f"    Val   - Low: {(y_val==0).sum()}, High: {(y_val==1).sum()}")
    print(f"    Test  - Low: {(y_test==0).sum()}, High: {(y_test==1).sum()}")
    
    return result

X_raw = labeled_rnaseq_df.drop('gene_id', axis=1).T.values  # (samples x genes)
y_binary = labeled_meta_df['risk_label'].values

data = stage_0_prepare_data(X_raw, y_binary, output_dir='deployment')


STAGE 0: Data Prepared
  Train: (1862, 19310) | Val: (400, 19310) | Test: (400, 19310)
  Label distribution:
    Train - Low: 781, High: 1081
    Val   - Low: 168, High: 232
    Test  - Low: 168, High: 232


In [13]:
for k in ['y_train', 'y_val', 'y_test']:
    data[k] = pd.to_numeric(data[k], errors='coerce').astype(int)

for k in ['X_train', 'X_val', 'X_test']:
    data[k] = np.nan_to_num(np.asarray(data[k], dtype=np.float32), nan=0.0, posinf=0.0, neginf=0.0)

print("Train classes:", np.unique(data['y_train'], return_counts=True))
print("Val classes:", np.unique(data['y_val'], return_counts=True))
print("Test classes:", np.unique(data['y_test'], return_counts=True))

Train classes: (array([0, 1]), array([ 781, 1081]))
Val classes: (array([0, 1]), array([168, 232]))
Test classes: (array([0, 1]), array([168, 232]))


In [10]:
# Random Forest 
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, classification_report
import joblib
import os

X_train, y_train = data['X_train'], data['y_train']
X_val, y_val = data['X_val'], data['y_val']
X_test, y_test = data['X_test'], data['y_test']

rf_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

# Validation
rf_val_pred = rf_model.predict(X_val)
rf_val_prob = rf_model.predict_proba(X_val)[:, 1]

print("=== Random Forest: VALIDATION ===")
print("Accuracy:", round(accuracy_score(y_val, rf_val_pred), 4))
print("Balanced Accuracy:", round(balanced_accuracy_score(y_val, rf_val_pred), 4))
print("F1:", round(f1_score(y_val, rf_val_pred), 4))
print("ROC AUC:", round(roc_auc_score(y_val, rf_val_prob), 4))

# Test
rf_test_pred = rf_model.predict(X_test)
rf_test_prob = rf_model.predict_proba(X_test)[:, 1]

print("\n=== Random Forest: TEST ===")
print("Accuracy:", round(accuracy_score(y_test, rf_test_pred), 4))
print("Balanced Accuracy:", round(balanced_accuracy_score(y_test, rf_test_pred), 4))
print("F1:", round(f1_score(y_test, rf_test_pred), 4))
print("ROC AUC:", round(roc_auc_score(y_test, rf_test_prob), 4))
print("\nClassification Report:\n")
print(classification_report(y_test, rf_test_pred, digits=4))


=== RANDOM FOREST: VALIDATION ===
Accuracy: 0.9237
Balanced Accuracy: 0.6812
F1: 0.9585
ROC AUC: 0.9206

=== RANDOM FOREST: TEST ===
Accuracy: 0.9198
Balanced Accuracy: 0.6935
F1: 0.9562
ROC AUC: 0.8616

Classification Report:

              precision    recall  f1-score   support

           0     0.8000    0.4000    0.5333        30
           1     0.9271    0.9871    0.9562       232

    accuracy                         0.9198       262
   macro avg     0.8636    0.6935    0.7447       262
weighted avg     0.9126    0.9198    0.9077       262


Saved: deployment/random_forest_model.pkl


In [14]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, classification_report
import joblib
import os

X_train, y_train = data['X_train'], data['y_train']
X_val, y_val = data['X_val'], data['y_val']
X_test, y_test = data['X_test'], data['y_test']

svm_model = SVC(
    kernel="linear",
    C=1.0,
    probability=True,
    class_weight="balanced",
    random_state=42
)

svm_model.fit(X_train, y_train)

# Validation
svm_val_pred = svm_model.predict(X_val)
svm_val_prob = svm_model.predict_proba(X_val)[:, 1]

print("=== Linear SVM: Validation ===")
print("Accuracy:", round(accuracy_score(y_val, svm_val_pred), 4))
print("Balanced Accuracy:", round(balanced_accuracy_score(y_val, svm_val_pred), 4))
print("F1:", round(f1_score(y_val, svm_val_pred), 4))
print("ROC AUC:", round(roc_auc_score(y_val, svm_val_prob), 4))

# Test
svm_test_pred = svm_model.predict(X_test)
svm_test_prob = svm_model.predict_proba(X_test)[:, 1]

print("\n=== Linear SVM: Test ===")
print("Accuracy:", round(accuracy_score(y_test, svm_test_pred), 4))
print("Balanced Accuracy:", round(balanced_accuracy_score(y_test, svm_test_pred), 4))
print("F1:", round(f1_score(y_test, svm_test_pred), 4))
print("ROC AUC:", round(roc_auc_score(y_test, svm_test_prob), 4))
print("\nClassification Report:\n")
print(classification_report(y_test, svm_test_pred, digits=4))

=== Linear SVM: Validation ===
Accuracy: 0.755
Balanced Accuracy: 0.7494
F1: 0.7879
ROC AUC: 0.8543

=== Linear SVM: Test ===
Accuracy: 0.7025
Balanced Accuracy: 0.6951
F1: 0.743
ROC AUC: 0.7852

Classification Report:

              precision    recall  f1-score   support

           0     0.6450    0.6488    0.6469       168
           1     0.7446    0.7414    0.7430       232

    accuracy                         0.7025       400
   macro avg     0.6948    0.6951    0.6949       400
weighted avg     0.7027    0.7025    0.7026       400


Saved: deployment/svm_model.pkl


In [16]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, roc_auc_score, classification_report
import joblib
import os

X_train, y_train = data['X_train'], data['y_train']
X_val, y_val = data['X_val'], data['y_val']
X_test, y_test = data['X_test'], data['y_test']

xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42
)

xgb_model.fit(X_train, y_train)

# Validation
xgb_val_pred = xgb_model.predict(X_val)
xgb_val_prob = xgb_model.predict_proba(X_val)[:, 1]

print("=== XGBOOST: Validation ===")
print("Accuracy:", round(accuracy_score(y_val, xgb_val_pred), 4))
print("Balanced Accuracy:", round(balanced_accuracy_score(y_val, xgb_val_pred), 4))
print("F1:", round(f1_score(y_val, xgb_val_pred), 4))
print("ROC AUC:", round(roc_auc_score(y_val, xgb_val_prob), 4))

# Test
xgb_test_pred = xgb_model.predict(X_test)
xgb_test_prob = xgb_model.predict_proba(X_test)[:, 1]

print("\n=== XGBOOST: Test ===")
print("Accuracy:", round(accuracy_score(y_test, xgb_test_pred), 4))
print("Balanced Accuracy:", round(balanced_accuracy_score(y_test, xgb_test_pred), 4))
print("F1:", round(f1_score(y_test, xgb_test_pred), 4))
print("ROC AUC:", round(roc_auc_score(y_test, xgb_test_prob), 4))
print("\nClassification Report:\n")
print(classification_report(y_test, xgb_test_pred, digits=4))

=== XGBOOST: Validation ===
Accuracy: 0.795
Balanced Accuracy: 0.7855
F1: 0.827
ROC AUC: 0.876

=== XGBOOST: Test ===
Accuracy: 0.7175
Balanced Accuracy: 0.7015
F1: 0.767
ROC AUC: 0.8054

Classification Report:

              precision    recall  f1-score   support

           0     0.6871    0.6012    0.6413       168
           1     0.7352    0.8017    0.7670       232

    accuracy                         0.7175       400
   macro avg     0.7111    0.7015    0.7041       400
weighted avg     0.7150    0.7175    0.7142       400

